# EXPERIMENT 6B – Solve the Taxi Problem using Reinforcement Learning

**Aim:** To solve the Taxi environment using Q-Learning. The taxi agent must pick up a passenger at a designated location and drop the passenger at the destination.

**Environment:** OpenAI Gymnasium Taxi-v3 (or compatible Gym Taxi-v3 environment).

**Algorithm:** Tabular Q-Learning.

## 1. Install and import libraries

If Gymnasium is not installed, run the installation command below once. The notebook uses the Taxi-v3 environment from Gymnasium.

In [ ]:
# In Jupyter, run this once if needed:
# %pip install gymnasium[toy-text]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

print("Gymnasium version:", gym.__version__)

## 2. Create the Taxi environment
Taxi-v3 has 500 possible states and 6 actions: move south, north, east, west, pickup and drop-off.

In [ ]:
# Create Taxi-v3 environment
env = gym.make("Taxi-v3")

# Read number of possible states and actions
state_size = env.observation_space.n
action_size = env.action_space.n

print("Number of states:", state_size)
print("Number of actions:", action_size)
print("Action meanings: 0=South, 1=North, 2=East, 3=West, 4=Pickup, 5=Dropoff")

## 3. Initialize the Q-table and learning parameters
Each row represents an environment state and each column represents an action. Q-values estimate the long-term reward of taking an action.

In [ ]:
# Initialize all Q-values to zero
Q = np.zeros((state_size, action_size))

# Q-Learning parameters
learning_rate = 0.7
discount_factor = 0.95
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.9995

episodes = 20000
max_steps = 200

episode_rewards = []

## 4. Train the Taxi agent
The agent explores with random actions and gradually learns to use actions with higher Q-values. Taxi gives a reward for successful drop-off and penalties for time steps and illegal pickup/drop-off actions.

In [ ]:
for episode in range(episodes):
    # Reset the environment at the beginning of each episode
    state, info = env.reset()
    total_reward = 0

    for step_number in range(max_steps):
        # Epsilon-greedy action selection
        if np.random.random() < epsilon:
            action = env.action_space.sample()  # Explore
        else:
            action = int(np.argmax(Q[state]))   # Exploit

        # Take action in the environment
        next_state, reward, terminated, truncated, info = env.step(action)

        # Q-Learning update
        best_next_action = np.max(Q[next_state])
        Q[state, action] = Q[state, action] + learning_rate * (
            reward + discount_factor * best_next_action - Q[state, action]
        )

        state = next_state
        total_reward += reward

        # End episode if task is complete or time limit is reached
        if terminated or truncated:
            break

    # Reduce exploration gradually
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    episode_rewards.append(total_reward)

print("Training completed.")
print("Final epsilon:", round(epsilon, 3))

## 5. Plot training rewards
The moving average shows whether the agent's reward is improving over time.

In [ ]:
# Calculate moving average over 500 episodes
window = 500
moving_average = np.convolve(
    episode_rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(9, 4))
plt.plot(moving_average)
plt.title("Taxi Q-Learning – Moving Average Reward")
plt.xlabel("Episode")
plt.ylabel("Average reward (500-episode window)")
plt.grid(True)
plt.show()

## 6. Test the trained agent
During testing, epsilon-greedy exploration is disabled. The agent always selects the action with the highest Q-value.

In [ ]:
# Create a separate environment for testing
test_env = gym.make("Taxi-v3", render_mode="ansi")

state, info = test_env.reset(seed=42)
total_reward = 0
test_actions = []
done = False

for step_number in range(100):
    # Choose the best learned action
    action = int(np.argmax(Q[state]))
    test_actions.append(action)

    next_state, reward, terminated, truncated, info = test_env.step(action)
    total_reward += reward
    state = next_state

    if terminated or truncated:
        done = True
        break

print("Test completed:", done)
print("Number of actions:", len(test_actions))
print("Action sequence:", test_actions)
print("Total test reward:", total_reward)

# Print final Taxi environment state
print(test_env.render())
test_env.close()

## 7. Evaluate over multiple test episodes
A single episode can vary depending on the initial passenger and destination. We test over 100 episodes and calculate the average reward and success rate.

In [ ]:
test_env = gym.make("Taxi-v3")
test_episodes = 100
test_rewards = []
successes = 0

for episode in range(test_episodes):
    state, info = test_env.reset()
    total_reward = 0

    for _ in range(200):
        action = int(np.argmax(Q[state]))
        state, reward, terminated, truncated, info = test_env.step(action)
        total_reward += reward

        if terminated:
            successes += 1
            break
        if truncated:
            break

    test_rewards.append(total_reward)

print("Average test reward:", round(np.mean(test_rewards), 2))
print("Successful drop-offs:", successes, "/", test_episodes)
print("Success rate:", round(successes / test_episodes * 100, 2), "%")
test_env.close()
env.close()

## 8. Conclusion

The Taxi-v3 problem was solved using tabular Q-Learning. The taxi agent learned to navigate to the passenger, pick them up, travel to the destination and drop them off. The learned policy was tested using multiple episodes.

**Result:** The training reward graph, test action sequence, average reward and success rate are displayed after running the notebook.